# CAMELYON16 → Phikon features on Colab

Runs `scripts/extract_all.py` on all 399 slides: download each slide to the VM's local disk, tile it, embed it with Phikon on the GPU, save the features to **Google Drive**, delete the slide.

- **Resumable.** Colab (free) disconnects after at most ~12 h. Just run all cells again: slides already on Drive are skipped.
- **Storage.** Features are saved as float16, about 12 GB in total. Check that your Drive has enough free space.
- Runtime → Change runtime type → **T4 GPU** before starting.

In [ ]:
# 1. Hardware: GPU, CPU count and local disk (slides are stored here temporarily)
!nvidia-smi -L
!nproc
!df -h /content | tail -1

In [ ]:
# 2. Google Drive: features go here, so they survive disconnections
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/camelyon16_phikon"
!mkdir -p {OUT}

In [ ]:
# 3. Code: project 2 from GitHub (+ project 1's `wsi` package, pinned to the same commit as uv.lock)
!git clone -q https://github.com/masnarivan/mil-camelyon16.git /content/mil-camelyon16 2>/dev/null || git -C /content/mil-camelyon16 pull -q
!pip install -q "wsi-embeddings @ git+https://github.com/masnarivan/wsi-embeddings@197c5151f1f2c1ff5bb7aeef340d7f682f2c0361"
!pip install -q --no-deps -e /content/mil-camelyon16

In [ ]:
# 4. Phikon weights into the local HF cache (the script itself runs offline)
from huggingface_hub import snapshot_download
snapshot_download("owkin/phikon")

In [ ]:
# 5. Extraction. Progress is printed one line per slide; the log is also on Drive.
!cd /content/mil-camelyon16 && python scripts/extract_all.py --split all --out {OUT} --raw /content/raw --dtype float16

In [ ]:
# 6. Status (can be run any time, also from a new session)
import pandas as pd, pathlib
done = len(list(pathlib.Path(OUT, "features").glob("*.npy")))
log = pd.read_csv(f"{OUT}/extract_log.csv")
last = log.drop_duplicates("slide_id", keep="last")
errors = last[last.error.notna() & (last.error != "")]
print(f"{done}/399 slides done, {len(errors)} with errors")
print(f"tiles so far: {last.n_tiles.sum():,.0f}, embedding speed: {last.n_tiles.sum() / last.t_embed_s.sum():.0f} tiles/s, "
      f"mean wait for download: {last.t_wait_download_s.mean():.0f} s/slide")
errors[["slide_id", "error"]]

When all 399 are done: download the `camelyon16_phikon` folder from Drive to the Mac
(`02-mil-camelyon16/outputs/`), then delete it from Drive.